# 164 — Seguridad de tools, MCP y supply chain

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** El contrato de veredicto (kind + evidence) modela el plano de **ejecución**: es
lo que una barrera de validación inspecciona antes de permitir que la herramienta actúe, registrando
evidencia auditable de la decisión.


In [ ]:
result = run_lab("safety", seed=164)
assert result["kind"] == "safety"
assert result["evidence"]
show(result)


**Ejercicio 2.**

```text
(a) supply_chain    (typosquatting de 'requests')
(b) tool_poisoning  (instrucción oculta en la descripción de la tool)
(c) confused_deputy (autoridad legítima ejercida por orden de un tercero no confiable)
(d) rug_pull        (comportamiento cambia tras la aprobación)
```

**Ejercicio 3.** Scope mínimo: acción `s3:GetObject` (solo lectura), recurso
`arn:aws:s3:::mi-bucket-docs/*` (un solo bucket), con credencial efímera y límite de tasa. Ataque
que se vuelve imposible: exfiltración por escritura o borrado (`PutObject`/`DeleteObject`) y acceso
a otros buckets de la cuenta, aunque el modelo sea engañado por inyección — el token simplemente no
puede escribir ni salir de ese prefijo.

**Ejercicio 4.** Campos mínimos por componente: nombre, versión exacta, hash/firma, licencia,
origen (repositorio) y relación (directa/transitiva). El día del CVE se consulta el SBOM para saber
si el componente afectado (y su versión) está presente y en qué servicios, priorizando el parche.
El pinning fija versiones pero no te dice *qué* tienes ni *dónde*; sin inventario no puedes
responder al alcance de una vulnerabilidad recién publicada.


In [ ]:
# Verificación del Ejercicio 2
casos = {"a": "supply_chain", "b": "tool_poisoning",
         "c": "confused_deputy", "d": "rug_pull"}
validas = {"confused_deputy", "tool_poisoning", "rug_pull", "supply_chain"}
assert set(casos.values()) == validas  # las cuatro amenazas, una por caso
print("clasificación correcta:", casos)


## Reflexión (guía)

1. Porque la obediencia del modelo es probabilística y el atacante trabaja para romperla; acotar la
   autoridad (token de solo lectura, límites, aprobación) hace que el daño sea imposible por
   construcción aunque el modelo falle.
2. El tool poisoning entrega la inyección por el *canal de metadatos de la herramienta* (su
   descripción), que el modelo consume automáticamente para decidir cómo usar la tool; la inyección
   clásica llega por el contenido/usuario. El vector de entrega difiere, el mecanismo es el mismo.
3. Visibilidad: el SBOM responde "qué componentes y versiones contengo y dónde", condición previa
   para saber si un CVE te afecta. El pinning garantiza reproducibilidad pero no inventario.
